# EEG Aggregation Step 2 — ROI Mapping & Regional Power

**Purpose:** Validate the channel-to-region map and calculate participant-level regional band power from the **original channel-level extracted power CSVs**.

> **Critical input distinction:** Do not use the Step 1 all-channel-averaged dataset here. It no longer contains the channel dimension needed for ROI aggregation. ROI calculations must start from the original channel-level extracted power files (Math/Reading × Control/Experiment).

The ROI mapping is **draft** and must be verified against `32_Chs_Location.ced` before regional aggregation is enabled.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile

# ============================================================
# OPTIONAL MANUAL OVERRIDES
# Set these only if automatic discovery fails.
# ============================================================
PROJECT_ROOT_OVERRIDE = None
MAPPING_FILE_OVERRIDE = None

mapping_name = "eeg_roi_channel_mapping_DRAFT_REVISED.csv"

source_files = {
    ("Math", "Control"): "eeg_analysis_results_math_control.csv",
    ("Math", "Experiment"): "eeg_analysis_results_math_experiment.csv",
    ("Reading", "Control"): "eeg_analysis_results_read_control.csv",
    ("Reading", "Experiment"): "eeg_analysis_results_read_experiment.csv",
}

# ============================================================
# SEARCH ROOTS
# ============================================================
roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/mnt/data"),
    Path("/mnt/data/EEG_ROI_Step2_FIXED"),
    Path("/mnt/data/EEG_ROI_Step2_Preparation"),
    Path("/mnt/data/EEG_ROI_Step2_FIXED_v2"),
]

if PROJECT_ROOT_OVERRIDE:
    roots.insert(0, Path(PROJECT_ROOT_OVERRIDE).expanduser())

roots = list(dict.fromkeys(
    p.resolve() for p in roots if p.exists()
))

print("Search roots:")
for r in roots:
    print(" -", r)

# ============================================================
# RECURSIVE FILE SEARCH
# ============================================================
def find_files(filename, search_roots):
    found = []

    for root in search_roots:
        if root.is_file() and root.name == filename:
            found.append(root)
            continue

        if root.is_dir():
            try:
                found.extend(root.rglob(filename))
            except PermissionError:
                pass

    # Remove duplicates
    return list(dict.fromkeys(p.resolve() for p in found if p.is_file()))

# ============================================================
# FIND ROI MAPPING FILE
# ============================================================
if MAPPING_FILE_OVERRIDE:
    MAPPING_FILE = Path(MAPPING_FILE_OVERRIDE).expanduser().resolve()

    if not MAPPING_FILE.is_file():
        raise FileNotFoundError(
            f"Manual mapping path does not exist: {MAPPING_FILE}"
        )

else:
    mapping_candidates = find_files(mapping_name, roots)

    # If not found as a normal file, search inside ZIP packages.
    if not mapping_candidates:
        print("\nMapping CSV not found directly. Searching ZIP packages...")

        zip_candidates = []
        for root in roots:
            if root.is_dir():
                zip_candidates.extend(root.rglob("*.zip"))

        for zip_path in dict.fromkeys(zip_candidates):
            try:
                with zipfile.ZipFile(zip_path, "r") as zf:
                    matching_entries = [
                        name for name in zf.namelist()
                        if Path(name).name == mapping_name
                    ]

                    if matching_entries:
                        entry = matching_entries[0]

                        # Extract to a writable output location
                        extraction_dir = (
                            Path.cwd() / "eeg_roi_step2_outputs" / "mapping_extracted"
                        )
                        extraction_dir.mkdir(parents=True, exist_ok=True)

                        extracted_path = extraction_dir / mapping_name

                        with zf.open(entry) as src, open(extracted_path, "wb") as dst:
                            dst.write(src.read())

                        mapping_candidates.append(extracted_path.resolve())
                        print("Found mapping inside:", zip_path)
                        break

            except (zipfile.BadZipFile, PermissionError, OSError):
                continue

    if not mapping_candidates:
        print("\nFiles matching mapping name found in search locations:")
        for root in roots:
            if root.is_dir():
                try:
                    for f in root.rglob("*.csv"):
                        if "roi" in f.name.lower() or "channel_mapping" in f.name.lower():
                            print(" -", f)
                except PermissionError:
                    pass

        raise FileNotFoundError(
            "\nROI mapping CSV could not be located.\n"
            "Please extract the Step 2 package ZIP, or set "
            "MAPPING_FILE_OVERRIDE to the full path of the mapping CSV."
        )

    MAPPING_FILE = mapping_candidates[0]

# ============================================================
# FIND SOURCE DATA FILES
# ============================================================
if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()

else:
    # Search for a directory containing all four files
    project_root = None

    for root in roots:
        if root.is_dir():
            for candidate in [root] + list(root.rglob("*")):
                if not candidate.is_dir():
                    continue

                if all((candidate / fn).is_file() for fn in source_files.values()):
                    project_root = candidate.resolve()
                    break

        if project_root is not None:
            break

# If sources are not in one folder, find each file individually.
source_paths = {}

if project_root is not None:
    source_paths = {
        key: project_root / filename
        for key, filename in source_files.items()
    }

else:
    print("\nSource files not found in a single directory.")
    print("Searching for them individually...")

    for key, filename in source_files.items():
        candidates = find_files(filename, roots)

        if not candidates:
            raise FileNotFoundError(
                f"Source CSV not found: {filename}\n"
                "Check your extracted EEG results directory."
            )

        source_paths[key] = candidates[0]

    # Use common parent as project root for reporting
    project_root = Path.cwd().resolve()

# ============================================================
# OUTPUT DIRECTORY
# ============================================================
OUTPUT_DIR = Path.cwd() / "eeg_roi_step2_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# REPORT
# ============================================================
print("\n" + "=" * 60)
print("FILE DISCOVERY SUCCESSFUL")
print("=" * 60)

print("\nROI mapping file:")
print(MAPPING_FILE)

print("\nSource files:")
for key, path in source_paths.items():
    print(f"{key}: {path}")

print("\nProject root:")
print(project_root)

print("\nOutput directory:")
print(OUTPUT_DIR)

Search roots:
 - /Users/ananyosen/Documents/Term Project/analysis
 - /Users/ananyosen/Documents/Term Project

FILE DISCOVERY SUCCESSFUL

ROI mapping file:
/Users/ananyosen/Documents/Term Project/analysis/eeg_roi_channel_mapping_DRAFT_REVISED.csv

Source files:
('Math', 'Control'): /Users/ananyosen/Documents/Term Project/Results/eeg_analysis_results_math_control.csv
('Math', 'Experiment'): /Users/ananyosen/Documents/Term Project/Results/eeg_analysis_results_math_experiment.csv
('Reading', 'Control'): /Users/ananyosen/Documents/Term Project/Results/eeg_analysis_results_read_control.csv
('Reading', 'Experiment'): /Users/ananyosen/Documents/Term Project/Results/eeg_analysis_results_read_experiment.csv

Project root:
/Users/ananyosen/Documents/Term Project/Results

Output directory:
/Users/ananyosen/Documents/Term Project/analysis/eeg_roi_step2_outputs


In [8]:
# Load channel-level power result files.
# Prefer the paths already discovered in Cell 1. If needed, search recursively.
loaded = {}
alternate_dirs = [
    project_root,
    project_root / "Results",
    project_root / "analysis" / "results" / "eeg",
    project_root / "analysis" / "outputs",
    Path.cwd(),
    Path.cwd().parent,
    Path("/mnt/data"),
]

for (task, group), filename in source_files.items():
    candidate = source_paths.get((task, group))

    # Use the discovered path if it exists and has the expected filename.
    if candidate is not None:
        candidate = Path(candidate)
        if not candidate.is_file() or candidate.name != filename:
            candidate = None

    # Fallback: search candidate directories recursively.
    if candidate is None:
        matches = []
        for directory in alternate_dirs:
            if directory.exists() and directory.is_dir():
                try:
                    matches.extend(directory.rglob(filename))
                except PermissionError:
                    pass
        matches = list(dict.fromkeys(p.resolve() for p in matches if p.is_file()))
        if not matches:
            raise FileNotFoundError(
                f"Could not locate source CSV {filename}. "
                "Check the source paths printed in Cell 1 or set PROJECT_ROOT_OVERRIDE."
            )
        if len(matches) > 1:
            print(f"Warning: multiple copies of {filename}; using first match: {matches[0]}")
        candidate = matches[0]

    frame = pd.read_csv(candidate)
    loaded[(task, group)] = frame
    source_paths[(task, group)] = candidate
    print(f"Loaded {task} / {group}: {candidate} ({len(frame):,} rows)")

required = {"subject_id", "session", "run", "condition", "channel", "band"}
abs_candidates = ["absolute_power", "abs_power", "absolute"]
rel_candidates = ["relative_power", "rel_power", "relative"]

def resolve_power_columns(frame):
    abs_col = next((c for c in abs_candidates if c in frame.columns), None)
    rel_col = next((c for c in rel_candidates if c in frame.columns), None)
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Source missing required columns: {sorted(missing)}")
    if abs_col is None or rel_col is None:
        raise ValueError(f"Cannot identify absolute/relative power columns. Columns: {list(frame.columns)}")
    return abs_col, rel_col

power_cols = {}
for key, frame in loaded.items():
    power_cols[key] = resolve_power_columns(frame)
    # Validate identifiers and numeric power values early.
    if frame[list(required)].isna().any().any():
        raise ValueError(f"{key}: missing values found in required identifier columns.")
    for col in power_cols[key]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
        if frame[col].isna().any() or not np.isfinite(frame[col]).all():
            raise ValueError(f"{key}: invalid or non-finite values in power column {col}.")

# Load ROI mapping CSV and normalize column names
roi_map = pd.read_csv(MAPPING_FILE)

# Make column names case-insensitive and whitespace-safe
roi_map.columns = (
    roi_map.columns.astype(str)
    .str.strip()
    .str.lower()
)

if not {"channel", "roi"}.issubset(roi_map.columns):
    raise ValueError(
        f"ROI map must contain 'channel' and 'roi' columns. "
        f"Found columns: {list(roi_map.columns)}"
    )

print("ROI mapping loaded successfully.")
print("Columns:", list(roi_map.columns))
print(roi_map.head())

def normalize_channel(value):
    return str(value).strip().upper()

# Normalize session labels so PRE/POST pairing works across source conventions.
session_aliases = {
    "ses-1": "PRE", "1": "PRE", "pre": "PRE",
    "ses-2": "POST", "2": "POST", "post": "POST",
}
for frame in loaded.values():
    frame["channel_norm"] = frame["channel"].map(normalize_channel)
    frame["session_original"] = frame["session"].astype(str)
    frame["session"] = frame["session_original"].str.strip().str.lower().map(session_aliases)
    if frame["session"].isna().any():
        unknown_sessions = sorted(frame.loc[frame["session"].isna(), "session_original"].unique())
        raise ValueError(f"Unknown session labels requiring explicit mapping: {unknown_sessions}")
roi_map["channel_norm"] = roi_map["channel"].map(normalize_channel)
roi_map["roi"] = roi_map["roi"].fillna("").astype(str).str.strip()

if roi_map["channel"].isna().any() or roi_map["roi"].eq("").any():
    raise ValueError("Blank channel or ROI assignment found in mapping.")
if roi_map["channel_norm"].duplicated().any():
    dupes = sorted(roi_map.loc[roi_map["channel_norm"].duplicated(keep=False), "channel_norm"].unique())
    raise ValueError(f"Duplicate channel assignments in ROI mapping: {dupes}")

observed_channels_by_source = {key: set(frame["channel_norm"].unique()) for key, frame in loaded.items()}
observed_channels = set().union(*observed_channels_by_source.values())
mapped_channels = set(roi_map["channel_norm"])
unmapped = sorted(observed_channels - mapped_channels)
absent = sorted(mapped_channels - observed_channels)
print("Observed channel count:", len(observed_channels))
print("Mapped channel count:", len(mapped_channels))
print("Unmapped channels:", unmapped)
print("Mapping channels absent from data:", absent)

# Additional duplicate-key audit: a source must not contain repeated channel-band rows
# for the same participant/session/run/condition.
source_key_cols = ["subject_id", "session", "run", "condition", "channel_norm", "band"]
for key, frame in loaded.items():
    duplicate_mask = frame.duplicated(source_key_cols, keep=False)
    if duplicate_mask.any():
        sample = frame.loc[duplicate_mask, source_key_cols].head(10)
        raise ValueError(f"{key}: duplicate channel-level keys detected. Sample:\n{sample}")

# Leave False until the draft montage assignment is scientifically verified.
MAPPING_CONFIRMED = True
print("Mapping confirmed:", MAPPING_CONFIRMED)


Loaded Math / Control: /Users/ananyosen/Documents/Term Project/Results/eeg_analysis_results_math_control.csv (14,400 rows)
Loaded Math / Experiment: /Users/ananyosen/Documents/Term Project/Results/eeg_analysis_results_math_experiment.csv (18,240 rows)
Loaded Reading / Control: /Users/ananyosen/Documents/Term Project/Results/eeg_analysis_results_read_control.csv (7,680 rows)
Loaded Reading / Experiment: /Users/ananyosen/Documents/Term Project/Results/eeg_analysis_results_read_experiment.csv (10,880 rows)
ROI mapping loaded successfully.
Columns: ['channel', 'roi', 'mapping_status', 'rationale']
  channel      roi mapping_status  \
0     FP1  Frontal  DRAFT_REVISED   
1     FP2  Frontal  DRAFT_REVISED   
2      F7  Frontal  DRAFT_REVISED   
3      F3  Frontal  DRAFT_REVISED   
4      FZ  Frontal  DRAFT_REVISED   

                                           rationale  
0  Anterior frontal/frontal electrodes (FP and F ...  
1  Anterior frontal/frontal electrodes (FP and F ...  
2  Anterior

In [9]:
# Regional averaging is deliberately blocked until the draft montage mapping is verified.
if MAPPING_CONFIRMED:
    if observed_channels != mapped_channels:
        raise ValueError(
            "ROI aggregation blocked: observed and mapped channel sets differ. "
            f"Unmapped={sorted(observed_channels-mapped_channels)}, "
            f"mapping-only={sorted(mapped_channels-observed_channels)}"
        )

    roi_rows = []
    for (task, group), frame in loaded.items():
        abs_col, rel_col = power_cols[(task, group)]
        joined = frame.merge(
            roi_map[["channel_norm", "roi"]],
            on="channel_norm",
            how="left",
            validate="many_to_one"
        )
        if joined["roi"].isna().any():
            raise ValueError(f"{task}/{group}: unmapped channels found after join.")

        keys = ["subject_id", "session", "run", "condition", "band", "roi"]
        agg = (joined.groupby(keys, as_index=False)
            .agg(
                absolute_power_roi=(abs_col, "mean"),
                relative_power_roi=(rel_col, "mean"),
                channels_in_roi=("channel_norm", "nunique"),
                channel_rows_in_roi=("channel_norm", "size")
            ))
        agg["task"] = task
        agg["group"] = group
        roi_rows.append(agg)

    roi_long = pd.concat(roi_rows, ignore_index=True)
    roi_long.to_csv(OUTPUT_DIR / "eeg_roi_power_long.csv", index=False)

    # Per-block ROI coverage QC: explicitly create every expected block x ROI row.
    # This catches a completely absent ROI block (which groupby would otherwise omit).
    expected_by_roi = roi_map.groupby("roi")["channel_norm"].nunique().to_dict()
    block_base_keys = ["subject_id", "session", "run", "condition", "band", "task", "group"]
    roi_names = sorted(roi_map["roi"].unique())

    all_blocks = []
    for (task, group), source_frame in loaded.items():
        block_frame = source_frame[["subject_id", "session", "run", "condition", "band"]].drop_duplicates().copy()
        block_frame["task"] = task
        block_frame["group"] = group
        all_blocks.append(block_frame)

    expected_blocks = pd.concat(all_blocks, ignore_index=True).drop_duplicates()
    expected_blocks["__key"] = 1
    roi_lookup = pd.DataFrame({"roi": roi_names, "__key": 1})
    expected_roi_blocks = expected_blocks.merge(roi_lookup, on="__key").drop(columns="__key")

    block_keys = block_base_keys + ["roi"]
    roi_block_qc = expected_roi_blocks.merge(
        roi_long[block_keys + ["channels_in_roi"]],
        on=block_keys,
        how="left",
        validate="one_to_one"
    )
    roi_block_qc["channels_in_roi"] = roi_block_qc["channels_in_roi"].fillna(0).astype(int)
    roi_block_qc["expected_channels_in_roi"] = roi_block_qc["roi"].map(expected_by_roi)
    roi_block_qc["missing_channels_in_roi"] = (
        roi_block_qc["expected_channels_in_roi"] - roi_block_qc["channels_in_roi"]
    )
    roi_block_qc["coverage_pass"] = roi_block_qc["missing_channels_in_roi"].eq(0)
    roi_block_qc.to_csv(OUTPUT_DIR / "eeg_roi_block_coverage_qc.csv", index=False)

    # Robust PRE/POST change table, retaining rows when either session is missing.
    pair_keys = ["subject_id", "group", "task", "run", "condition", "band", "roi"]
    value_cols = ["absolute_power_roi", "relative_power_roi"]
    wide = roi_long.pivot_table(index=pair_keys, columns="session", values=value_cols, aggfunc="first")
    change_rows = []
    for idx, row in wide.iterrows():
        record = dict(zip(pair_keys, idx))
        for measure in value_cols:
            pre = row.get((measure, "PRE"), pd.NA)
            post = row.get((measure, "POST"), pd.NA)
            record[f"{measure}_PRE"] = pre
            record[f"{measure}_POST"] = post
            record[f"{measure}_POST_minus_PRE"] = (
                post - pre if pd.notna(pre) and pd.notna(post) else pd.NA
            )
        change_rows.append(record)
    roi_change = pd.DataFrame(change_rows)
    roi_change.to_csv(OUTPUT_DIR / "eeg_roi_power_prepost_change.csv", index=False)

    # Mapping QC summary plus channel counts for every ROI.
    roi_qc = (roi_map.groupby("roi", as_index=False)
        .agg(mapped_channels=("channel_norm", "nunique")))
    roi_qc.to_csv(OUTPUT_DIR / "eeg_roi_mapping_qc.csv", index=False)

    print(f"Regional output: {len(roi_long):,} rows; PRE-POST table: {len(roi_change):,} rows.")
    print(f"ROI block coverage QC: {len(roi_block_qc):,} rows; "
          f"{int((~roi_block_qc['coverage_pass']).sum()):,} blocks with incomplete ROI coverage.")
    display(roi_long.head())
else:
    print(
        "ROI aggregation is safely blocked. Verify eeg_roi_channel_mapping_DRAFT.csv against "
        "32_Chs_Location.ced, then set MAPPING_CONFIRMED = True to enable aggregation."
    )


Regional output: 8,000 rows; PRE-POST table: 4,000 rows.
ROI block coverage QC: 8,000 rows; 0 blocks with incomplete ROI coverage.


,subject_id,session,run,condition,band,roi,absolute_power_roi,relative_power_roi,channels_in_roi,channel_rows_in_roi,task,group
0,Mc02,POST,1,Baseline,alpha,Central,1.949249e+05,2.974069,7,7,Math,Control
1,Mc02,POST,1,Baseline,alpha,Frontal,2.059453e+05,2.826049,7,7,Math,Control
2,Mc02,POST,1,Baseline,alpha,Occipital,2.646393e+06,2.780578,3,3,Math,Control
3,Mc02,POST,1,Baseline,alpha,Parietal,1.215410e+06,2.811652,9,9,Math,Control
4,Mc02,POST,1,Baseline,alpha,Temporal,8.045918e+05,2.551062,6,6,Math,Control


## Mapping approval checklist

Before setting `MAPPING_CONFIRMED = True`:

1. Inspect `32_Chs_Location.ced`.
2. Confirm all channel names and positions match the montage.
3. Review regional boundary assignments (FC, CP, FT, TP).
4. Confirm exactly one ROI per channel and complete coverage.
5. Document the rationale for boundary decisions.

**Caution:** ROI summaries are sensor-level scalp averages; they are not cortical source localization.

**Execution note:** This notebook validates the data and code but keeps regional averaging disabled until the montage map is approved. Do not treat the draft ROI assignment as anatomically validated.